# One-way interaction prediction on the role-aware line graph

In [ ]:
# ---------------- settings ----------------
import os
DATA_CSV   = "Data_msystems/Features.csv"          # adjust if the data is elsewhere
OUT_DIR    = "results"
LABELS     = ["effect>0", "threshold_0.66"]          # run both labels; remove one to run only the other
THRESHOLD  = 0.66                                   # used by "threshold_0.66": scaled effect >= 0.66 -> positive
EPOCHS     = int(os.environ.get("EPOCHS", 3000))
MODELS     = ["GAT-SAGE", "GAT", "GraphSAGE", "CGNN"]
ABLATION   = ["GAT-SAGE"]                           # models also run on the role-agnostic graph ([] to skip)
HIDDEN, HEADS, LR, SEED = 100, 4, 1e-3, 42
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import dgl.nn as dglnn
import dgl.function as fn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

device = torch.device("cpu")
if torch.cuda.is_available():
    try:
        dgl.graph(([0], [0])).to("cuda")          # does this DGL build support the GPU?
        device = torch.device("cuda")
    except Exception:
        print("WARNING: PyTorch sees a GPU, but this DGL build is CPU-only -> running on CPU (slow).")
print("device:", device, "| torch", torch.__version__, "| CUDA", torch.version.cuda, "| dgl", dgl.__version__)

## 1. Data, label, features, folds

In [ ]:
df = pd.read_csv(DATA_CSV)
df = df[df["Carbon"] != "Water"].reset_index(drop=True)

effect = df["2 on 1: Effect"].values
effect_scaled = (effect - effect.min()) / (effect.max() - effect.min())
Y = {"effect>0": (effect > 0).astype(int),
     "threshold_0.66": (effect_scaled >= THRESHOLD).astype(int)}
raw_cut = effect.min() + THRESHOLD * (effect.max() - effect.min())
for lab in LABELS:
    extra = f"  (= raw effect >= {raw_cut:.4f})" if lab == "threshold_0.66" else ""
    print(f"label {lab:15s}: positive {100 * Y[lab].mean():.1f}%{extra}")

# the 13 features
FEATURE_COLS = list(df.columns[5:18])
assert len(FEATURE_COLS) == 13 and "2 on 1: Effect" not in FEATURE_COLS, FEATURE_COLS
X = MinMaxScaler().fit_transform(df[FEATURE_COLS].values)

# one group per co-culture (unordered species pair + carbon): both directions stay in the same fold
groups = df.apply(lambda r: "_".join(sorted([str(r["Bug 1"]), str(r["Bug 2"])])) + "_" + str(r["Carbon"]), axis=1).values
FOLDS_BY_LABEL = {lab: list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, Y[lab], groups=groups))
                  for lab in LABELS}
print("nodes:", len(df), "| co-cultures:", len(set(groups)), "| self-pairs:", int((df["Bug 1"] == df["Bug 2"]).sum()))
print("features:", FEATURE_COLS)

## 2. Line graphs
**Role-aware:** connected if, under the same carbon condition, they share the same effector (`Bug 2`) or the same affected species (`Bug 1`).

**Role-agnostic (ablation):** connected if they share any species under the same carbon condition.

In [ ]:
def pairs_within(groups_index):
    src, dst = [], []
    for idx in groups_index:
        idx = np.asarray(idx)
        if len(idx) > 1:
            u, v = np.meshgrid(idx, idx)
            keep = u != v
            src.append(u[keep]); dst.append(v[keep])
    return np.unique(np.stack([np.concatenate(src), np.concatenate(dst)], 1), axis=0)   # no duplicate edges

def make_graph(edges):
    return dgl.graph((torch.tensor(edges[:, 0]), torch.tensor(edges[:, 1])), num_nodes=len(df))

# role-aware: same effector or same affected, same carbon
ra_groups = [idx for role in ["Bug 2", "Bug 1"] for idx in df.groupby([role, "Carbon"]).indices.values()]
EDGES = {"role-aware": pairs_within(ra_groups)}

# role-agnostic: share any species, same carbon
long = pd.concat([pd.DataFrame({"node": df.index, "species": df["Bug 1"], "Carbon": df["Carbon"]}),
                  pd.DataFrame({"node": df.index, "species": df["Bug 2"], "Carbon": df["Carbon"]})])
long = long.drop_duplicates()
EDGES["role-agnostic"] = pairs_within(long.groupby(["species", "Carbon"])["node"].apply(np.array).values)

GRAPHS = {k: make_graph(e) for k, e in EDGES.items()}
for k, gk in GRAPHS.items():
    print(f"{k:14s}: nodes {gk.num_nodes()} | directed edges {gk.num_edges()} | undirected {gk.num_edges() // 2}")

## 3. Ollivier–Ricci curvature (for CGNN only)
Computed once and saved to `results/curvature_one-way.csv`; later runs reuse the file.

In [ ]:
def add_curvature(gname):
    edges = EDGES[gname]
    curv_file = f"{OUT_DIR}/curvature_one-way_{gname}.csv"
    if os.path.exists(curv_file):
        curv = pd.read_csv(curv_file)
    else:
        from GraphRicciCurvature.OllivierRicci import OllivierRicci
        import networkx as nx
        t0 = time.time()
        G = nx.Graph(); G.add_nodes_from(range(len(df)))
        G.add_edges_from(map(tuple, edges[edges[:, 0] < edges[:, 1]]))
        orc = OllivierRicci(G, alpha=0.5, verbose="ERROR"); orc.compute_ricci_curvature()
        curv = pd.DataFrame([(u, v, d["ricciCurvature"]) for u, v, d in orc.G.edges(data=True)], columns=["u", "v", "curvature"])
        curv.to_csv(curv_file, index=False)
        print(f"curvature ({gname}) computed in {time.time() - t0:.0f}s")
    lookup = {}
    for u, v, c in curv[["u", "v", "curvature"]].itertuples(index=False):
        lookup[(int(u), int(v))] = c; lookup[(int(v), int(u))] = c
    r = np.array([lookup.get((int(u), int(v)), 0.0) for u, v in edges])
    deg = np.zeros(len(df)); np.add.at(deg, edges[:, 1], r)
    deg = np.where(deg == 0, 1.0, deg)
    GRAPHS[gname].edata["w"] = torch.tensor(r / np.sqrt(np.abs(deg[edges[:, 0]] * deg[edges[:, 1]])), dtype=torch.float32)
    print(f"curvature ({gname}): mean {r.mean():.4f} | min {r.min():.4f} | max {r.max():.4f}")

if "CGNN" in MODELS:
    add_curvature("role-aware")
if "CGNN" in ABLATION:
    add_curvature("role-agnostic")

## 4. Models (two layers, hidden size 100)

In [ ]:
class GAT_SAGE(nn.Module):                # GAT layer (4 heads) -> GraphSAGE layer
    def __init__(s, i, h, o, heads=HEADS):
        super().__init__()
        s.gat1 = dglnn.GATConv(i, h, num_heads=heads, activation=F.relu)
        s.conv2 = dglnn.SAGEConv(h * heads, o, aggregator_type="mean")
    def forward(s, g, x, return_hidden=False):
        h = s.gat1(g, x).flatten(1); out = s.conv2(g, h)
        return (out, h) if return_hidden else out

class GAT(nn.Module):                     # two GAT layers
    def __init__(s, i, h, o, heads=HEADS):
        super().__init__()
        s.gat1 = dglnn.GATConv(i, h, num_heads=heads, activation=F.elu)
        s.gat2 = dglnn.GATConv(h * heads, o, num_heads=1)
    def forward(s, g, x, return_hidden=False):
        h = s.gat1(g, x).flatten(1); out = s.gat2(g, h).mean(1)
        return (out, h) if return_hidden else out

class SAGE(nn.Module):                    # two GraphSAGE layers (mean aggregation)
    def __init__(s, i, h, o):
        super().__init__()
        s.conv1 = dglnn.SAGEConv(i, h, aggregator_type="mean")
        s.conv2 = dglnn.SAGEConv(h, o, aggregator_type="mean")
    def forward(s, g, x, return_hidden=False):
        h = F.relu(s.conv1(g, x)); out = s.conv2(g, h)
        return (out, h) if return_hidden else out

class CGNNLayer(nn.Module):               # curvature-weighted message passing
    def __init__(s, i, o):
        super().__init__(); s.linear = nn.Linear(i, o)
    def forward(s, g, x):
        with g.local_scope():
            g.ndata["h"] = x
            g.update_all(lambda e: {"m": e.data["w"].unsqueeze(1) * e.src["h"]}, fn.sum("m", "h_neigh"))
            return s.linear(g.ndata["h_neigh"])

class CGNN(nn.Module):
    def __init__(s, i, h, o):
        super().__init__(); s.l1 = CGNNLayer(i, h); s.l2 = CGNNLayer(h, o)
    def forward(s, g, x, return_hidden=False):
        h = F.relu(s.l1(g, x)); out = s.l2(g, h)
        return (out, h) if return_hidden else out

BUILD = {"GAT-SAGE": lambda: GAT_SAGE(13, HIDDEN, 2), "GAT": lambda: GAT(13, HIDDEN, 2),
         "GraphSAGE": lambda: SAGE(13, HIDDEN, 2), "CGNN": lambda: CGNN(13, HIDDEN, 2)}

## 5. Grouped 5-fold cross-validation (metrics on validation nodes only)

In [ ]:
def fold_metrics(name, k, true, prob):
    pred = (prob >= 0.5).astype(int)
    return {"model": name, "fold": k, "accuracy": accuracy_score(true, pred),
            "precision": precision_score(true, pred, zero_division=0), "recall": recall_score(true, pred, zero_division=0),
            "f1": f1_score(true, pred, zero_division=0), "roc_auc": roc_auc_score(true, prob),
            "pr_auc": average_precision_score(true, prob)}

def run_gnn(name, y, FOLDS, gname="role-aware", lab=""):
    gd = GRAPHS[gname].to(device)
    xd = torch.tensor(X, dtype=torch.float32, device=device); yd = torch.tensor(y, device=device)
    rows, cm, emb = [], np.zeros((2, 2), dtype=int), []
    for k, (tr, va) in enumerate(FOLDS, start=1):
        t0 = time.time(); torch.manual_seed(SEED + k)
        tm = torch.zeros(len(y), dtype=torch.bool, device=device); tm[tr] = True
        vm = torch.zeros(len(y), dtype=torch.bool, device=device); vm[va] = True
        model = BUILD[name]().to(device); opt = torch.optim.Adam(model.parameters(), lr=LR)
        for ep in range(EPOCHS):
            model.train()
            loss = F.cross_entropy(model(gd, xd)[tm], yd[tm])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            out, hid = model(gd, xd, return_hidden=True)
            prob = F.softmax(out[vm], 1)[:, 1].cpu().numpy()
        true = y[va]                                           # VALIDATION nodes only
        rows.append(fold_metrics(name, k, true, prob))
        cm += confusion_matrix(true, (prob >= 0.5).astype(int), labels=[0, 1])
        emb.append((hid[vm].cpu().numpy().astype(np.float16), true, np.full(len(true), k)))
        print(f"  [{lab}] {name} [{gname}] fold {k}: F1 {rows[-1]['f1']:.4f} | {time.time()-t0:.0f}s")
    tag = f"{lab}_{name}_{gname}"
    np.savez_compressed(f"{OUT_DIR}/embeddings_one-way_{tag}.npz", h=np.concatenate([e[0] for e in emb]),
                        label=np.concatenate([e[1] for e in emb]), fold=np.concatenate([e[2] for e in emb]))
    return pd.DataFrame(rows), cm

## 6. Run everything for each label (GNNs and role-agnostic ablation)

In [ ]:
ALL = {}            # ALL[label] = {"results": {...}, "cms": {...}, "abl": {...}}
for lab in LABELS:
    y, FOLDS = Y[lab], FOLDS_BY_LABEL[lab]
    RESULTS, CMS, ABL = {}, {}, {}
    for name in MODELS:
        print(f"\n=== [{lab}] {name} ===")
        RESULTS[name], CMS[name] = run_gnn(name, y, FOLDS, lab=lab)
        RESULTS[name].to_csv(f"{OUT_DIR}/folds_one-way_{lab}_{name}_role-aware.csv", index=False)
    for name in ABLATION:
        print(f"\n=== [{lab}] {name} (role-agnostic) ===")
        ABL[name], _ = run_gnn(name, y, FOLDS, "role-agnostic", lab=lab)
        ABL[name].to_csv(f"{OUT_DIR}/folds_one-way_{lab}_{name}_role-agnostic.csv", index=False)
    ALL[lab] = {"results": RESULTS, "cms": CMS, "abl": ABL}

## 7. Summary per label (mean ± std over the 5 folds)

In [ ]:
METRICS = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]
ORDER = ["GAT-SAGE", "GAT", "GraphSAGE", "CGNN"]
for lab in LABELS:
    R, A = ALL[lab]["results"], ALL[lab]["abl"]
    order = [m for m in ORDER if m in R]
    summary = pd.DataFrame({m: [f"{R[m][c].mean():.4f} ± {R[m][c].std(ddof=0):.4f}" for c in METRICS] for m in order},
                           index=METRICS).T
    for name in A:
        summary.loc[f"{name} (role-agnostic)"] = [f"{A[name][c].mean():.4f} ± {A[name][c].std(ddof=0):.4f}" for c in METRICS]
    summary.to_csv(f"{OUT_DIR}/table1_one-way_{lab}_summary.csv")
    print(f"\n########## label: {lab} ##########")
    display(summary)

## 8. Table 1 in LaTeX, one per label (best value per column in bold)

In [ ]:
names = {"accuracy": "Accuracy", "precision": "Precision", "recall": "Recall", "f1": "F1-score",
         "roc_auc": "ROC-AUC", "pr_auc": "PR-AUC"}
label_text = {"effect>0": "a measurement is positive if its effect is $>0$",
              "threshold_0.66": "a measurement is positive if its min--max scaled effect is $\\geq 0.66$"}
for lab in LABELS:
    R = ALL[lab]["results"]; order = [m for m in ORDER if m in R]
    best = {c: max(order, key=lambda m: R[m][c].mean()) for c in METRICS}
    def cell(m, c):
        s = f"{R[m][c].mean():.4f} $\\pm$ {R[m][c].std(ddof=0):.4f}"
        return f"\\textbf{{{s}}}" if best[c] == m else s
    lines = [r"\begin{table*}[t]", r"\centering", r"\small",
             r"\caption{Performance comparison of role-aware GNN architectures "
             rf"for one-way interaction prediction ({label_text[lab]}; mean $\pm$ standard deviation, $n = 5$ folds). The best value in each column is shown in bold.}}",
             r"\label{tab:oneway_comparison}", r"\begin{tabular}{l" + "c" * len(METRICS) + "}", r"\toprule",
             "Model & " + " & ".join(names[c] for c in METRICS) + r" \\", r"\midrule"]
    lines += [f"{m} & " + " & ".join(cell(m, c) for c in METRICS) + r" \\" for m in order]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]
    latex = "\n".join(lines)
    open(f"{OUT_DIR}/table1_one-way_{lab}.tex", "w").write(latex)
    print(f"\n%%%%%%%%%% label: {lab} %%%%%%%%%%\n" + latex)

## 9. Paired t-tests per label (same folds): GAT-SAGE vs the other GNNs, role-aware vs role-agnostic

In [ ]:
from scipy.stats import ttest_rel
def compare(a, b, la, lb):
    for c in ["f1", "pr_auc"]:
        t, p = ttest_rel(a[c], b[c])
        print(f"  {la:24s} vs {lb:14s} {c:7s}: diff {a[c].mean() - b[c].mean():+.4f}  p = {p:.4f}")
def gnn_tests(R, A):
    if "GAT-SAGE" in R:
        for m in [x for x in R if x != "GAT-SAGE"]:
            compare(R["GAT-SAGE"], R[m], "GAT-SAGE", m)
    for name in A:
        compare(R[name], A[name], f"{name} role-aware", "role-agnostic")
for lab in LABELS:
    print(f"\n########## label: {lab} ##########")
    gnn_tests(ALL[lab]["results"], ALL[lab]["abl"])

## 10. Confusion matrices normalised by the total
All four cells together sum to 100%; the number in brackets is the count summed over the 5 validation folds. Saved as `results/confusion_one-way_<label>.pdf`.

In [ ]:
import matplotlib.pyplot as plt
for lab in LABELS:
    C = ALL[lab]["cms"]; order = [m for m in ORDER if m in C]
    fig, axes = plt.subplots(1, len(order), figsize=(3.0 * len(order), 3.1))
    for ax, m in zip(np.atleast_1d(axes), order):
        counts = C[m]; pct = counts / counts.sum() * 100                      # normalised by the total: all cells sum to 100%
        ax.imshow(pct, cmap="Blues", vmin=0, vmax=pct.max())
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{pct[i, j]:.1f}%\n({counts[i, j]})", ha="center", va="center", fontsize=9,
                        color="white" if pct[i, j] > 0.6 * pct.max() else "black")
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["Negative", "Positive"]); ax.set_yticklabels(["Negative", "Positive"])
        ax.set_title(m, fontsize=11); ax.set_xlabel("Predicted")
    np.atleast_1d(axes)[0].set_ylabel("True")
    fig.suptitle(f"One-way, label: {lab} (normalised by total, all cells sum to 100%)", y=1.03)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_one-way_{lab}.pdf", bbox_inches="tight"); plt.show()

## 11. t-SNE of hidden embeddings (validation nodes only)

In [ ]:
from sklearn.manifold import TSNE
for lab in LABELS:
    show = [m for m in ORDER if m in ALL[lab]["results"]]
    fig, axes = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3.3))
    for ax, m in zip(np.atleast_1d(axes), show):
        d = np.load(f"{OUT_DIR}/embeddings_one-way_{lab}_{m}_role-aware.npz")
        idx = np.random.default_rng(SEED).choice(len(d["label"]), size=min(4000, len(d["label"])), replace=False)
        z = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(d["h"][idx].astype(np.float32))
        lbl = d["label"][idx]
        for l_, col, txt in [(0, "#9aa7b5", "negative"), (1, "#d0312d", "positive")]:
            ax.scatter(z[lbl == l_, 0], z[lbl == l_, 1], s=3, c=col, label=txt, alpha=0.6)
        ax.set_title(m); ax.set_xticks([]); ax.set_yticks([])
    np.atleast_1d(axes)[0].legend(markerscale=4, frameon=False)
    fig.suptitle(f"label: {lab}", y=1.03)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/tsne_one-way_{lab}.pdf", bbox_inches="tight"); plt.show()